In [93]:
# Import Basic Packgaes 
import numpy as np
import pandas as pd
import datetime
import statsmodels as sm
import itertools
import glob
import os
from scipy.signal import argrelextrema
from scipy.signal import find_peaks, peak_widths
import scipy as sp

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns # advanced vizs
from pandas.plotting import lag_plot

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

# Set Color Palettes
palette1 = itertools.cycle(sns.color_palette(palette='Set1'))
palette2 = itertools.cycle(sns.color_palette(palette='Set2'))

In [94]:
# Load the csv files

df = pd.read_csv("D:\\TDMS\\6.Pump_1_Right12'oclock crak left good\\CSV Files\\4_P1_R12clockcrackLgood_16Oct23.csv")
df.head()

,Time,Cycle-detect,P1-CPM,P1-Water Suction Pressure,P1-Water Discharge Pressure,P1-Channel-3,P1-Air-Supply-pressure,P1-Water Suction Flowrate,P1-Water Discharge Flowrate,P1-Air Supply Flowrate
0,2023-10-16 00:00:00.035,0.0,56.435687,5.202183,50.014067,24.167140,104.509756,0.0,79.565668,84.876388
1,2023-10-16 00:00:00.044,0.0,56.435687,5.314246,48.893333,23.816910,105.490398,0.0,79.565668,84.941553
2,2023-10-16 00:00:00.074,0.0,56.435687,6.420869,44.550489,22.486039,113.265490,0.0,79.826250,85.332544
3,2023-10-16 00:00:00.104,0.0,56.435687,5.356270,37.475856,21.365305,115.647050,0.0,79.478807,85.202213
4,2023-10-16 00:00:00.134,0.0,56.435687,6.532932,33.623333,19.964387,114.946591,0.0,79.565668,85.397709


In [95]:
clean_df = df[df['P1-Air-Supply-pressure'] > 90]

In [96]:
clean_df.reset_index(inplace=True, drop=True)

In [97]:
clean_df['newtime'] = pd.date_range('2023-10-16 00:00:00.000', freq='30ms', periods=len(clean_df))

In [98]:
clean_df['newtime'] =pd.to_datetime(clean_df['newtime'])

In [99]:
clean_df.set_index(['newtime'], inplace=True)

In [100]:
# Define FFT parameters
dt = 0.03
f1, f2 = 1000, 1000  # Hz
diff = 100

In [101]:
# Define the time interval for slicing
start_time = clean_df.index[0]
end_time = clean_df.index[-1]
time_interval = pd.Timedelta(hours=1)

In [102]:
current_time = start_time
while current_time <= end_time:
    next_time = current_time + time_interval

    # Slice the DataFrame based on the specified time range
    sliced_df = clean_df.iloc[(clean_df.index >= current_time) & (clean_df.index <= next_time)]

    # Perform FFT analysis only if there is data in the sliced DataFrame
    n = len(sliced_df)
    if n > 0:
        t = sliced_df.index
        s = sliced_df['P1-Air-Supply-pressure']

        # FFT
        s -= s.mean()
        fr = np.fft.rfftfreq(n, dt)
        fou = np.fft.rfft(s)

        # Plot and save the FFT plot
        plt.figure(figsize=(28, 10))
        plt.subplot(211)
        plt.plot(t, s)
        plt.title('Data with Noise')

        plt.subplot(212)
        plt.plot(np.fft.fftshift(fr), np.fft.fftshift(np.abs(fou)))
        plt.xticks(np.arange(min(np.fft.fftshift(fr)), max(np.fft.fftshift(fr)), 0.3))
        plt.ylim(0, 40000)
        plt.title('Spectrum of Noisy Data')

        # Save the plot with a meaningful filename
        filename = os.path.join("D:\\6.Pump_1_Right12'oclock crak left good_FFT\\Cleaned Data_FFT_pictures\\04.P1_16Oct23_FFT_Pictures", f'{current_time.strftime("%H-%M-%S")}_{next_time.strftime("%H-%M-%S")}_FFT.png')
        plt.savefig(filename)
        plt.close()
            #fr_list = [np.fft.fftshift(fr)]
            #amp_list = [np.fft.fftshift(np.abs(fou))]
            #fourier_df = pd.DataFrame(data=[fr_list, amp_list],columns=['frequency','amplidute'])
            #print(fourier_df.loc[fourier_df['amplidute'].idxmax()])

    else:
        # Handle the case where the DataFrame is empty (e.g., print a message)
        print(f"No data in the time range {current_time} - {next_time}")

    current_time = next_time

In [103]:
freq_amp_dfs = []

In [104]:
current_time = start_time
while current_time <= end_time:
    
    next_time = current_time + time_interval
    
    # Slice the DataFrame based on the specified time range
    sliced_df = clean_df.iloc[(clean_df.index >= current_time) & (clean_df.index <= next_time)]
    
    # Perform FFT analysis only if there is data in the sliced DataFrame
    n = len(sliced_df)
    if n > 0:
        t = sliced_df.index
        s = sliced_df['P1-Air-Supply-pressure']

        # FFT
        s -= s.mean()
        fr = np.fft.rfftfreq(n, dt)
        fou = np.fft.rfft(s)
        
        # Create a DataFrame for the current fr,amp
        temp_df = pd.DataFrame({'Frequency': fr,'Amplitude': np.abs(fou)})

        # Append the DataFrame to the list
        freq_amp_dfs.append(temp_df)
        
    else:
        # Handle the case where the DataFrame is empty (e.g., print a message)
        print(f"No data in the time range {current_time} - {next_time}")

    current_time = next_time    

In [105]:
# Concatenate all DataFrames in the list into a single DataFrame
freq_amp_df = pd.concat(freq_amp_dfs, ignore_index=True)

In [106]:
# Save the result DataFrame to a CSV file
freq_amp_df.to_csv(r"D:\\6.Pump_1_Right12'oclock crak left good_FFT\\Cleaned Data_FFT_df\\04.P1_16Oct23_fr,amp.csv")

In [107]:
entropy_list = []

In [108]:
current_time = start_time
while current_time <= end_time:
        
    next_time = current_time + time_interval
        
    # Slice the DataFrame based on the specified time range
    sliced_df = clean_df.iloc[(clean_df.index >= current_time) & (clean_df.index <= next_time)]
        
    # Perform FFT analysis only if there is data in the sliced DataFrame
    # Calculate the probabilities
    asp_prob =  sliced_df['P1-Air-Supply-pressure'].value_counts()/sum(sliced_df['P1-Air-Supply-pressure'].value_counts().values)
        
    # Calculate the entropy
    asp_entropy = sp.stats.entropy(asp_prob, base=2)
            
    # Create a DataFrame for the current fr,amp
    #temp_df = pd.DataFrame(asp_entropy,columns=['Entropy'])

    entropy_list.append(asp_entropy)
            
    current_time = next_time    

In [109]:
Sh_En = pd.DataFrame(entropy_list,columns=['Entropy'])

In [110]:
# Save the result DataFrame to a CSV file
Sh_En.to_csv(r"D:\\6.Pump_1_Right12'oclock crak left good_FFT\\Cleaned Data_Shannon Entropy\\04.P1_16Oct23_shannon_val.csv")